# Phase 2 Part B: Classical U-Net Deep Semantic Segmentation
## High-Resolution Cartosat-1 (2.5m) Panchromatic Urban Land-Use & Road Extraction

### Objectives
1. **Architecture**: Symmetric Convolutional Encoder-Decoder with Skip Connections (DoubleConv, MaxPool, UpConv, 1x1 Conv Head).
2. **5 Urban Classes**: Water (0), Vegetation (1), Barren (2), Built-up (3, including railway terminals), Roads (4, pure road network).
3. **Road-Prioritized Loss**: Compound Focal Loss + Multi-Class Dice Loss ($w_{\text{road}} = 3.0$).
4. **Benchmarking**: Compare U-Net against SegFormer (Vision Transformer) on the identical 246 Cartosat-1 patch dataset.

In [ ]:
import os
import sys
from pathlib import Path

# Add local scripts to path
scripts_dir = Path.cwd() / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Inspect Patches and Class Alignment

In [ ]:
import pandas as pd

patches_root = Path.cwd().parents[1] / 'Phase_1_Data_Prep_and_Labeling' / 'data' / 'patches'
df = pd.read_csv(patches_root / 'dataset_index.csv')
print(f'Total Patches: {len(df)}')
print(df['split'].value_counts())
df.head()

## 2. Model Architecture & Loss Formulation

In [ ]:
from model import build_unet
from loss import CompoundRoadPrioritizedLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_unet(num_classes=5).to(device)
criterion = CompoundRoadPrioritizedLoss(num_classes=5, road_weight=3.0).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net Trainable Parameters: {total_params:,}')

## 3. Train U-Net on Cartosat-1 Patches

In [ ]:
from train import train_unet

output_dir = Path.cwd()
results = train_unet(
    patches_dir=patches_root,
    output_dir=output_dir,
    epochs=10,
    batch_size=4,
    learning_rate=2e-4,
)

## 4. Visual Evaluation and Comparison

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display training curves
curve_path = output_dir / 'training_curves.png'
if curve_path.exists():
    plt.figure(figsize=(14, 5))
    plt.imshow(Image.open(curve_path))
    plt.axis('off')
    plt.show()

# Display sample test predictions
test_vis_dir = output_dir / 'visual_results' / 'test_visuals'
vis_files = list(test_vis_dir.glob('*.png'))[:3] if test_vis_dir.exists() else []
for vf in vis_files:
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(vf))
    plt.axis('off')
    plt.title(vf.name)
    plt.show()